In [1]:
from pathlib import Path
import os

BASE_DIR = Path("Data/")

DIRS = {
    "data": BASE_DIR / "data",
    "cache": BASE_DIR / "cache",
    "models": BASE_DIR / "models",
    "faiss": BASE_DIR / "faiss",
    "checkpoints": BASE_DIR / "checkpoints",
    "outputs": BASE_DIR / "outputs",
    "logs": BASE_DIR / "logs",
    "wordnet": BASE_DIR / "wordnet",
    "retrieval": BASE_DIR / "retrieval",
}

for directories in DIRS.values():
    directories.mkdir(parents=True, exist_ok=True)

print("PROJECT DIRECTORIES ESTABLISHED")
print(BASE_DIR)

PROJECT DIRECTORIES ESTABLISHED
Data


In [2]:
# %pip install transformers datasets sentence-transformers faiss-cpu seqeval accelerate -q
# %pip install -q faiss-cpu nltk wikipedia
# %pip install -q torch torchvision torchaudio

In [3]:
# SUPPORT IMPORTS
import os
import json
import random
import pickle
import warnings

# UTILITY IMPORTS
import faiss                    
from tqdm import tqdm 
from pathlib import Path

# DATA ANALYSIS IMPORTS
import numpy as np
import pandas as pd

# DATA VISUALISATION IMPORTS
import matplotlib.pyplot as mlt
import seaborn as sns
import plotly.express as px

# DATA LOADING IMPORTS
import torch
import wikipedia

# MODEL IMPORTS
from datasets import load_dataset, load_from_disk, DatasetDict

# TRANSFORMER IMPORTS
from transformers import (
    Trainer,
    AutoTokenizer,
    TrainingArguments,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    pipeline
)
from sentence_transformers import SentenceTransformer, CrossEncoder

# NATURAL LANGUAGE PROCESSING IMPORTS
import nltk

# DATABASE CORPUS IMPORTS
from nltk.corpus import wordnet

# METRICS IMPORTS
from seqeval.metrics import classification_report, f1_score
import evaluate

In [4]:
# SETTINGS
warnings.filterwarnings('ignore')
%matplotlib inline

In [5]:
# CONSTANTS AND DECLARATIONS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TOP = 5
SEED = 42
SENTENCES = 15
MAX_LENGTH = 256

DOMAINS = [
    # MEDICAL FIELDS AND CONCEPTS
    "Pharmacology", "Immunology", "Antibiotics", "Drug interaction", "Neurology",
    # DISEASES
    "Hypertension", "Asthma", "Anxiety disorder", "Insomnia", "Depression",
    # COMMON DRUGS
    "Paracetamol", "Adderall", "Ibuprofen", "Amoxicillin", "Aspirin",
    "Lisinopril", "Azithromycin", "Doxycycline", "Insulin"
]


BASE_DIR = Path("Data/")
DATASET_DIR = BASE_DIR/"data"/"conll2003"
SPLIT_DIR = BASE_DIR/"data"/"conll2003_SPLIT"
KB_PATH = BASE_DIR/"retrieval"/"knowledge_base.pkl"
INDEX_PATH = BASE_DIR/"faiss"/"wiki.index"
EMBED_PATH = BASE_DIR/"faiss"/"wiki_embeddings.npy"
ENRICHED_PATH = BASE_DIR/"data"/"Enriched"
TOKENIZED_PATH = BASE_DIR/"data"/"Tokenised_dataset"
CHECKPOINT_DIR = BASE_DIR/"checkpoints"
MODEL_PATH = BASE_DIR/"models"/"FINAL_MODEL"


BASE_MODEL = "dslim/bert-base-NER"
RETRIEVER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



In [6]:
# DATASET AND DATABASE 
# nltk.download("wordnet")
# nltk.download("omw-1.4")

# DOWNLOAD DATASET
if DATASET_DIR.exists():
    DATASET = load_from_disk(str(DATASET_DIR))
    print("LOADING CACHED DATASET.....")
else:
    DATASET = load_dataset("conll2003")
    DATASET.save_to_disk(str(DATASET_DIR))
    print("DOWNLOADED & CACHED DATASET")

# SPLIT DATASET
if SPLIT_DIR.exists():
     DATASET = load_from_disk(str(SPLIT_DIR))
     print("LOADING SPLIT DATASET......")
else:
    TRAIN_TEST = DATASET["train"].train_test_split(
        test_size=0.25,
        train_size=0.75,
        seed=SEED
    )

    DATASET = DatasetDict({
        "train" : TRAIN_TEST["train"],
        "test": TRAIN_TEST["test"]
    })

    DATASET.save_to_disk(str(SPLIT_DIR))
    print("SPLITED & CACHED DATASET")


# DOWNLOAD WIKICORPUS
if KB_PATH.exists():
    with open(KB_PATH, "rb") as FILE:
        KNOWLEDGE_BASE = pickle.load(FILE)
    print("LOADING KNOWLEDGE BASE.....")
else:
    TOPICS = DOMAINS
    KNOWLEDGE_BASE = []
    for topic in tqdm(TOPICS):
        try:
            TEXT = wikipedia.summary(topic, sentences=SENTENCES)
            KNOWLEDGE_BASE.append(TEXT)
        except:
            print(f"ERROR: {Exception}")
    with open(KB_PATH, "wb") as FILE:
        pickle.dump(KNOWLEDGE_BASE, FILE)
    print("LOADED & CACHED KNOWLEDGE BASE")

LOADING CACHED DATASET.....
LOADING SPLIT DATASET......
LOADING KNOWLEDGE BASE.....


In [7]:
# LABEL CONFIGURATION
LABELS_ = DATASET["train"].features["ner_tags"].feature.names
print(LABELS_)

ID2LABEL = {i:label for i,label in enumerate(LABELS_)}
LABEL2ID = {label: i for i, label in enumerate(LABELS_)}

print(f"ID2LABEL: {ID2LABEL}")
print(f"LABEL2ID: {LABEL2ID}")

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
ID2LABEL: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}
LABEL2ID: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


In [8]:
# LOADING BASE DISTIL MODEL

BASE_TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL)
BASE_BERT = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels = len(LABELS_),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

print("BASE DISTIL* MODEL LOADED")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BASE DISTIL* MODEL LOADED


In [9]:
# LOAD RETRIEVER MODELS

RETRIEVER = SentenceTransformer(
    RETRIEVER_MODEL,
    device=DEVICE
) 
CROSS_ENCODER = CrossEncoder(
    ENCODER_MODEL,
    device=DEVICE
)

print("RETRIEVER MODEL LOADED")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RETRIEVER MODEL LOADED


In [14]:
def EXPAND_ENTITIES(ENTITY_):
    EXPANSIONS = set()
    for SYNSET in wordnet.synsets(ENTITY_):
        for LEMMA in SYNSET.lemmas():
            EXPANSIONS.add(LEMMA.name().replace("_"," "))
        for HYPER in SYNSET.hypernyms():
            for LEMMA in HYPER.lemmas():
                EXPANSIONS.add(LEMMA.name().replace("_"," "))
        for HYPO in SYNSET.hyponyms():
            for LEMMA in HYPO.lemmas():
                EXPANSIONS.add(LEMMA.name().replace("_"," "))
    return list(EXPANSIONS)

def RETRIEVE_KNOWLEDGE(QUERY, INDEX, TOP=TOP):
    QUERY_EMBEDDING = RETRIEVER.encode(
        [QUERY],
        convert_to_numpy=True,
    )
    
    faiss.normalize_L2(QUERY_EMBEDDING)
    SCORES, INDICES = INDEX.search(QUERY_EMBEDDING, TOP)
    RETRIEVED_DOCS = [KNOWLEDGE_BASE[IDX] for IDX in INDICES[0]]
    PAIRS = [[QUERY, DOC] for DOC in RETRIEVED_DOCS]
    RERANK_SCORES = CROSS_ENCODER.predict(PAIRS)
    BEST_DOC = RETRIEVED_DOCS[np.argmax(RERANK_SCORES)]
    return BEST_DOC

def DETECT_ENTITIES(NER_PIPELINE, TEXT):
    outputs = NER_PIPELINE(TEXT)
    return outputs


# ENTITY AWARE SCALING FUNCTION
#                                                   Xi = (ei√d + (1-ei)1/√d) * Xi
def SCALER(EMBEDDINGS, ENTITY_MASK):
    DIMENSION = EMBEDDINGS.shape[-1]
    SCALED_EMBEDDINGS = []
    for EMBED, MASK in zip(EMBEDDINGS, ENTITY_MASK):
        if MASK == 1:
            SCALED = EMBED * np.sqrt(DIMENSION)
        else:
            SCALED = EMBED * (1/np.sqrt(DIMENSION))
        SCALED_EMBEDDINGS.append(SCALED)
    return np.array(SCALED_EMBEDDINGS)


In [11]:
# KNOWLEDGE INDEXING
if INDEX_PATH.exists() and EMBED_PATH.exists():
    INDICES = faiss.read_index(str(INDEX_PATH))
    EMBEDDINGS = np.load(str(EMBED_PATH))
    print("LOADING INDICES & EMBEDDINGS...")
else:
    EMBEDDINGS = RETRIEVER.encode(
        KNOWLEDGE_BASE,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    DIMENSION = EMBEDDINGS.shape[1]
    INDICES = faiss.IndexFlatIP(DIMENSION)
    faiss.normalize_L2(EMBEDDINGS)
    INDICES.add(EMBEDDINGS)    
    faiss.write_index(INDICES, str(INDEX_PATH))
    np.save(str(EMBED_PATH), EMBEDDINGS)
    print("KNOWLEDGE EMBEDDED & INDEXED")

LOADING INDICES & EMBEDDINGS...


In [12]:
## PIPELINES
# BASE NER PIPELINE
NER_PIPELINE = pipeline(
    "ner",
    model=BASE_BERT,
    tokenizer=BASE_TOKENIZER,
    aggregation_strategy="simple",
    device=0 if DEVICE=="cuda" else 1
)

# CONTEXT ENRICHMENT PIPELINE
def ENRICHER(TOKENS):
    SENTENCE = " ".join(TOKENS)
    ENTITIES = DETECT_ENTITIES(NER_PIPELINE, SENTENCE)
    RETRIEVED_CONTEXTS = []
    for ENTIRY in ENTITIES:
        WORD = ENTIRY["word"]
        EXPANSIONS = EXPAND_ENTITIES(WORD)
        QUERY = WORD + " " + " ".join(EXPANSIONS[:7])
        KNOWLEDGE = RETRIEVE_KNOWLEDGE(QUERY, INDICES)
        RETRIEVED_CONTEXTS.append(
            f"ENTITY: {WORD} CONTEXT: {KNOWLEDGE}"
        )
    ENRICHED_QUERY = SENTENCE + " " + " ".join(RETRIEVED_CONTEXTS)
    return ENRICHED_QUERY


In [13]:
DATASET

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 10530
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3511
    })
})

In [15]:
if ENRICHED_PATH.exists():
    ENRICHED = load_from_disk(str(ENRICHED_PATH))
    print("LOADING ENRICHED DATASET...")
else:
    def RICHIE(DATA):
        DATA["Enriched_Text"] = ENRICHER(DATA["tokens"])
        return DATA
    
    ENRICHED = DATASET.map(
        RICHIE,
        desc="Enriching...", 
        disable_nullable=False
    )
    ENRICHED.save_to_disk(str(ENRICHED_PATH))
    print("SAVED & CACHED ENRICHED DATASET")

Parameter 'function'=<function RICHIE at 0x000001AF40801120> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Enriching...:   0%|          | 0/10530 [00:00<?, ? examples/s]

Enriching...:   0%|          | 0/3511 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10530 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3511 [00:00<?, ? examples/s]

SAVED & CACHED ENRICHED DATASET


In [16]:
# TOKENISATION AND LABEL ALIGNMENT 
def ALIGNER(DATA):
    TOKENISED_INPUTS = BASE_TOKENIZER(
        DATA["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )
    LABELS = []
    for i, lable in enumerate(DATA["ner_tags"]):
        WORD_IDS = TOKENISED_INPUTS.word_ids(batch_index=i)
        PREVIOUS_WORD_IDX = None
        LABEL_IDS = []
        for WORD_IDX in WORD_IDS:
            if WORD_IDX is None:
                LABEL_IDS.append(-100)
            elif WORD_IDX != PREVIOUS_WORD_IDX:
                LABEL_IDS.append(lable[WORD_IDX])
            else:
                LABEL_IDS.append(-100)
            PREVIOUS_WORD_IDX = WORD_IDX
        LABELS.append(LABEL_IDS)
    TOKENISED_INPUTS["labels"] = LABELS
    return TOKENISED_INPUTS

In [17]:
if TOKENIZED_PATH.exists():
    TOKENISED_DATASET = load_from_disk(str(TOKENIZED_PATH))
    print("LOADING TOKENISED DATASET....")
else:
    TOKENISED_DATASET = ENRICHED.map(
        ALIGNER,
        batched=True
    )
    TOKENISED_DATASET.save_to_disk(str(TOKENIZED_PATH))
    print("LOADED & CACHED TOKENISED DATSET")

Map:   0%|          | 0/10530 [00:00<?, ? examples/s]

Map:   0%|          | 0/3511 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10530 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3511 [00:00<?, ? examples/s]

LOADED & CACHED TOKENISED DATSET


In [20]:
REFINED_MODEL = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS_),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

METRIC = evaluate.load("seqeval")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
def COMPUTE_METRICS(PAIR):
    PREDICTIONS, LABELS = PAIR
    PREDICTIONS = np.argmax(PREDICTIONS, axis=2)
    TRUE_PREDICTIONS = [
        [
            LABELS_[p]
            for(p, l) in zip(PREDICTION, LABEL)
            if l != -100
        ]
        for PREDICTION, LABEL in zip(PREDICTIONS, LABELS_)
    ]
    TRUE_LABELS = [
        [
            LABELS_[l]
            for(p,l) in zip(PREDICTION, LABEL)
            if l != -100
        ]
        for PREDICTION, LABEL in zip(PREDICTIONS, LABELS_)
    ]
    RESULTS = METRIC.compute(
        predictions=TRUE_PREDICTIONS,
        references=TRUE_LABELS
    )
    return {
        "precision": RESULTS["overall_precision"],
        "recall": RESULTS["overall_recall"],
        "f1": RESULTS["overall_f1"],
        "accuracy": RESULTS["overall_accuracy"],
    }

In [23]:
training_args = TrainingArguments(
    output_dir=str(BASE_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir=str(BASE_DIR / "logs"),
    logging_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [25]:
DATA_COLLECTOR = DataCollatorForTokenClassification(BASE_TOKENIZER)

TRAINER = Trainer(
    model=REFINED_MODEL,
    args=training_args,
    train_dataset=TOKENISED_DATASET["train"],
    eval_dataset=TOKENISED_DATASET["test"],
    data_collator=DATA_COLLECTOR,
    compute_metrics=COMPUTE_METRICS
)

In [ ]:
CHECKPOINTS = list(CHECKPOINT_DIR.glob("checkpoint-*"))

if len(CHECKPOINTS) > 0:
    LAST = sorted(
        CHECKPOINTS,
        key=lambda checkpoint: int(str(checkpoint).split("-")[-1])
    )
    print(f"RESUME TRAINING FROM {LAST}")
    TRAINER.train(
        resume_from_checkpoint=str(LAST)
    )
else:
    TRAINER.train()

Epoch,Training Loss,Validation Loss


In [ ]:
TRAINER.save_model(str(MODEL_PATH))
BASE_TOKENIZER.save_pretrained(str(MODEL_PATH))

In [ ]:
RESULTS = TRAINER.evaluate()
print(f"FINAL RESULTS: {RESULTS}")

with open(BASE_DIR / "outputs" / "evaluation.json", "w") as f:
    json.dump(RESULTS, f, indent=4)

In [ ]:
FINAL_PIPELINE = pipeline(
    "ner",
    model=MODEL_PATH,
    tokenizer=MODEL_PATH,
    aggregation_strategy="simple",
    device=0 if DEVICE == "cuda" else -1
)

In [ ]:
def PREDICT(TEXT):
    ENRICHED = ENRICHER(TEXT.aplit())
    OUPUTS = FINAL_PIPELINE(ENRICHED)
    return OUPUTS

TEST = "Barack Obama visited Microsoft headquarters in Seattle where he was treated by Paracetamol."
PREDICTIONS = PREDICT(TEST)

PREDICTIONS

In [ ]:
PIPELINE_INFO = {
    "base_model": BASE_MODEL,
    "retriever_model": "sentence-transformers/all-MiniLM-L6-v2",
    "cross_encoder": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "dataset": "CoNLL2003"
}

with open(BASE_DIR / "outputs" / "pipeline_config.json", "w") as f:
    json.dump(PIPELINE_INFO, f, indent=4)